## Background

Surrogate optimization replaces expensive simulations with trained neural networks inside
an optimization model (Misener & Biegler, 2023). For MPC — where optimization repeats at
every control step — the conventional ReLU + MILP approach (Grimstad & Andersson, 2019;
OMLT, Ceccon et al., 2022) incurs significant overhead per solve.

This notebook explores a different approach: **smooth activations (Softplus)** embedded as
Gurobi nonlinear constraints, optimized with the **nonlinear barrier solver**. The key identity
of this work is *not* the neural network itself — it is the use of smooth differentiable
surrogates with continuous NLP methods, moving beyond the ReLU/MILP paradigm.

> *Can smooth neural surrogates be optimised efficiently in a repeated MPC loop?*

The workflow is:
$$\underbrace{\text{BSM2 simulator}}_{\text{IWA benchmark}} \;\longrightarrow\; \text{Training data} \;\longrightarrow\; \underbrace{\text{Smooth surrogate}}_{\text{Softplus NN}} \;\longrightarrow\; \underbrace{\text{MPC optimise}}_{\text{Gurobi NLP barrier}}$$

This is a simplified educational demonstration inspired by neural-network-based wastewater MPC
workflows (Ramesh et al., 2026) and surrogate-based process optimization (Durkin & Guo, 2023;
Wu et al., 2024). Unlike those works, which use piecewise-linear or integer-programming
formulations, this notebook uses smooth Softplus surrogates with Gurobi's nonlinear solver.

The surrogate approximates the one-step bioreactor transition:
$$x_{t+1} \approx f_\theta(x_t,\, u_t)$$
where $x_t = [\text{DO},\; \text{NH}_4,\; \text{NO}_3]$ (dissolved oxygen, ammonia,
nitrate in reactor 5) and $u_t = \text{KLa}_5$ (aeration intensity). Wastewater bioreactor
dynamics are highly nonlinear, continuous, and repeatedly optimized — smooth surrogate models
are a natural fit for barrier-based MPC.

**Note on local optimality:** The nonlinear barrier solver finds *locally* optimal solutions.
This is typical of real-time NLP-based MPC practice and acceptable for control applications.

**References:**
Ceccon et al. (2022) [OMLT](https://arxiv.org/abs/2202.02414) ·
Grimstad & Andersson (2019) [arXiv](https://arxiv.org/abs/1907.03140) ·
Misener & Biegler (2023) [doi](https://doi.org/10.1016/j.compchemeng.2023.108411) ·
Tsay (2021) [doi](https://doi.org/10.1016/j.compchemeng.2021.107452) ·
Ramesh et al. (2026) ·
Durkin & Guo (2023) [doi](https://doi.org/10.1016/j.resconrec.2023.106930) ·
Wu et al. (2024) ·
[Gurobi ML](https://gurobi-machinelearning.readthedocs.io/en/stable/) ·
[Gurobi nonlinear](https://docs.gurobi.com/projects/optimizer/en/current/features/nonlinear.html) ·
[BSM benchmarks](https://github.com/wwtmodels/Benchmark-Simulation-Models) ·
[BSM2 Python](https://github.com/FAU-EVT/bsm2-python)

In [ ]:
!uv pip install bsm2-python

In [ ]:
import logging
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import gurobipy as gp
from gurobipy import GRB
from gurobi_ml import add_predictor_constr
from bsm2_python import BSM2OL

# Suppress BSM2 INFO logging for cleaner notebook output
logging.getLogger().setLevel(logging.WARNING)

## Step 1 — Data from the BSM2 Wastewater Simulator

Data is generated from the [BSM2 Python](https://github.com/FAU-EVT/bsm2-python) package,
a Python implementation of the IWA Benchmark Simulation Model 2. BSM2 implements the
full ASM1 bioreactor model across five reactors, primary clarifier, settler, and sludge
treatment. We extract three key state variables from reactor 5 (the main aerobic zone):
dissolved oxygen (DO), ammonia (NH₄), and nitrate (NO₃). Control is aeration intensity
KLa₅.

A smooth neural surrogate is trained on BSM2 trajectories to approximate $x_{t+1} = f(x_t, u_t)$
and embedded into Gurobi for MPC optimisation. The surrogate is trained offline; the BSM2
simulator acts as the real plant during closed-loop MPC.

In [ ]:
# --- Offline data generation from BSM2 simulator ---
print("Stabilising BSM2 (IWA benchmark)... (~20 s)")
bsm_data = BSM2OL()
bsm_data.stabilize()
print("Stabilised. Generating training trajectories...")

# Random-walk exploration of KLa5 (aeration in reactor 5)
# Reactors 1-4 held at default values; only reactor 5 is varied.
N_DATA = 2000
np.random.seed(42)
kla5 = 120.0
X_list, y_list = [], []

for i in range(N_DATA):
    kla5 = np.clip(kla5 + np.random.randn() * 30, 0.0, 240.0)
    klas = np.array([0.0, 0.0, 0.0, 120.0, kla5])

    # State before step: (DO, NH4, NO3) from reactor 5 effluent
    x_t = np.array([bsm_data.y_out5[7], bsm_data.y_out5[9], bsm_data.y_out5[8]])
    bsm_data.step(i, klas)
    x_t1 = np.array([bsm_data.y_out5[7], bsm_data.y_out5[9], bsm_data.y_out5[8]])

    X_list.append([*x_t, kla5])
    y_list.append(x_t1)

X_raw = np.array(X_list, dtype=np.float32)  # (N, 4): [DO, NH4, NO3, KLa5]
y_raw = np.array(y_list, dtype=np.float32)  # (N, 3): [DO_next, NH4_next, NO3_next]
print(f"{N_DATA} (state, control) → next-state pairs generated.")
for i, name in enumerate(["DO", "NH₄", "NO₃", "KLa5"]):
    print(f"  {name}: [{X_raw[:, i].min():.2f}, {X_raw[:, i].max():.2f}]")

## Step 2 — Smooth Neural Surrogate

`nn.Sequential` with `nn.Softplus` — embedded by `gurobi-machinelearning` as Gurobi
nonlinear constraints. This preserves smooth differentiability and lets the nonlinear
barrier exploit gradient information, unlike ReLU/MILP branch-and-bound.

In [ ]:
X_mean = X_raw.mean(0)
X_std = X_raw.std(0) + 1e-9
y_mean = y_raw.mean(0)
y_std = y_raw.std(0) + 1e-9
X_norm = (X_raw - X_mean) / X_std
y_norm = (y_raw - y_mean) / y_std

torch.manual_seed(42)
surrogate = nn.Sequential(
    nn.Linear(4, 24),
    nn.Softplus(beta=2.0),
    nn.Linear(24, 24),
    nn.Softplus(beta=2.0),
    nn.Linear(24, 3),
)
loader = DataLoader(
    TensorDataset(torch.tensor(X_norm), torch.tensor(y_norm)),
    batch_size=128,
    shuffle=True,
)
opt = torch.optim.Adam(surrogate.parameters(), lr=1e-3)
surrogate.train()
for _ in range(400):
    for xb, yb in loader:
        opt.zero_grad()
        nn.MSELoss()(surrogate(xb), yb).backward()
        opt.step()

surrogate.eval()
with torch.no_grad():
    yp = surrogate(torch.tensor(X_norm)).numpy() * y_std + y_mean
rmse = np.sqrt(np.mean((yp - y_raw) ** 2, axis=0))
print(
    f"Surrogate RMSE — DO: {rmse[0]:.4f}  NH₄: {rmse[1]:.4f}  NO₃: {rmse[2]:.4f}  mg/L"
)

## Step 3 — MPC Closed-Loop

The MPC controller uses the smooth surrogate for predictions; the BSM2 simulator acts as
the real plant. At each step, the controller solves a 6-step lookahead problem, applies the
first action to BSM2, and repeats — the standard receding-horizon principle.

Objective: minimise aeration energy + penalty for low DO (oxygen deficit) and high NH₄
(ammonia violation). The surrogate is embedded via `add_predictor_constr` for each horizon
step.

In [ ]:
# --- Initialise real plant (separate BSM2 instance) ---
print("Stabilising BSM2 plant for MPC simulation... (~20 s)")
bsm_mpc = BSM2OL()
bsm_mpc.stabilize()
print("Ready. Running MPC closed-loop...")

N_STEPS, H_MPC, DT = 48, 6, 0.25  # 12 h at 15-min steps, 6-step horizon

cl_x = np.zeros((N_STEPS + 1, 3))  # closed-loop state trajectory
cl_u = np.zeros(N_STEPS)  # closed-loop aeration trajectory
cl_x[0] = [bsm_mpc.y_out5[7], bsm_mpc.y_out5[9], bsm_mpc.y_out5[8]]
mpc_times = []

for t in range(N_STEPS):
    xc = cl_x[t]

    # --- Build MPC optimisation problem ---
    m = gp.Model()
    m.Params.OutputFlag = 0
    m.Params.NonConvex = 2
    m.Params.OptimalityTarget = 1
    m.Params.TimeLimit = 10.0

    u_v = m.addMVar(H_MPC, lb=0.0, ub=240.0, name="u")
    x_v = m.addMVar((H_MPC, 3), lb=[0.0, 0.0, 0.0], ub=[8.0, 50.0, 30.0], name="x")
    ni = m.addMVar((H_MPC, 4), lb=-10.0, ub=10.0, name="ni")
    no = m.addMVar((H_MPC, 3), lb=-10.0, ub=10.0, name="no")

    for k in range(H_MPC):
        x_prev = xc if k == 0 else x_v[k - 1]
        for j in range(3):
            val = float(xc[j]) if k == 0 else x_v[k - 1, j]
            m.addConstr(ni[k, j] == (val - float(X_mean[j])) / float(X_std[j]))
        m.addConstr(ni[k, 3] == (u_v[k] - float(X_mean[3])) / float(X_std[3]))
        add_predictor_constr(m, surrogate, ni[k : k + 1, :], no[k : k + 1, :])
        for j in range(3):
            m.addConstr(x_v[k, j] == no[k, j] * float(y_std[j]) + float(y_mean[j]))

    # Soft constraint penalties: DO >= 2 mg/L, NH4 <= 4 mg/L
    ds = m.addMVar(H_MPC, lb=0.0)
    ns = m.addMVar(H_MPC, lb=0.0)
    for k in range(H_MPC):
        m.addConstr(ds[k] >= 2.0 - x_v[k, 0])
        m.addConstr(ns[k] >= x_v[k, 1] - 4.0)

    m.setObjective(
        1e-4 * gp.quicksum(u_v[k] * DT for k in range(H_MPC))
        + gp.quicksum(ds[k] + ns[k] for k in range(H_MPC)),
        GRB.MINIMIZE,
    )
    u_v.Start = np.full(H_MPC, cl_u[t - 1] if t > 0 else 120.0)

    t0 = time.time()
    m.optimize()
    mpc_times.append(time.time() - t0)

    if m.Status in (GRB.OPTIMAL, GRB.LOCALLY_OPTIMAL, GRB.TIME_LIMIT):
        u_opt = float(u_v.X[0])
    else:
        u_opt = cl_u[t - 1] if t > 0 else 120.0

    # Apply first action to real BSM2 plant
    klas = np.array([0.0, 0.0, 0.0, 120.0, u_opt])
    bsm_mpc.step(t, klas)
    cl_u[t] = u_opt
    cl_x[t + 1] = [bsm_mpc.y_out5[7], bsm_mpc.y_out5[9], bsm_mpc.y_out5[8]]

    if t % 12 == 0:
        print(
            f"  t={t * DT:.1f}h: KLa={u_opt:.0f}  DO={cl_x[t + 1, 0]:.2f}  NH4={cl_x[t + 1, 1]:.2f}"
        )

print(f"Done. Mean MPC solve time: {np.mean(mpc_times):.3f}s per step")

## Results

In [ ]:
tv = np.arange(N_STEPS + 1) * DT
tc = np.arange(N_STEPS) * DT

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
axes[0].plot(tv, cl_x[:, 0], lw=2)
axes[0].axhline(2.0, color="k", linestyle="--", alpha=0.6, label="Target ≥2 mg/L")
axes[0].set_ylabel("DO (mg/L)")
axes[0].legend(fontsize=8)
axes[0].set_title("Wastewater MPC — Softplus Surrogate (NLP barrier, BSM2 plant)")

axes[1].plot(tv, cl_x[:, 1], color="tab:orange", lw=2)
axes[1].axhline(4.0, color="red", linestyle="--", alpha=0.7, label="Limit ≤4 mg-N/L")
axes[1].set_ylabel("NH₄ (mg-N/L)")
axes[1].legend(fontsize=8)

axes[2].plot(tv, cl_x[:, 2], color="tab:green", lw=2)
axes[2].set_ylabel("NO₃ (mg-N/L)")

axes[3].step(tc, cl_u, where="post", color="tab:purple", lw=2)
axes[3].set_ylabel("KLa₅ (h⁻¹)")
axes[3].set_xlabel("Time (h)")

plt.tight_layout()
plt.show()

viol = np.maximum(cl_x[1:, 1] - 4.0, 0)
print(
    f"Mean solve: {np.mean(mpc_times):.3f}s  |  "
    f"NH₄ violation: {(viol > 0).mean() * 100:.0f}%  |  Max: {viol.max():.2f} mg-N/L"
)

*Copyright © 2026 Gurobi Optimization, LLC — Apache 2.0 License*